# 🧬 NB01 — Joint DAE + Discrete-Hazard Survival Engine (v1.0-ram · Multi-Cohort)
**Stage:** NB01 | **RAM strategy:** sequential cohorts, float32, del+gc per fold

---

### What this notebook does
Trains a two-phase Joint DAE + Discrete-Hazard survival model on multiple TCGA
cohorts (LGG, KIRC, LUAD).  Evaluates with bootstrapped C-index CIs and exports
latent embeddings for downstream use.

**Pipeline stages:**

| Phase | Cell | Task |
|-------|------|------|
| Setup | 1 | Imports · seeding · device |
| Config | 2 | config.yaml → canonical paths |
| Data | 3 | Load cohort · variance filter · split · scale |
| Model | 4 | Dataset · Architecture · Loss |
| Train | 5 | Phase 1 DAE pretraining · Phase 2 joint fine-tune |
| Eval | 6 | Bootstrap C-index · Kaplan-Meier |
| Export | 7 | Checkpoint + latent embeddings |

**RAM strategy (mirrors NB00/NB02):**

| Technique | Where |
|-----------|-------|
| `del data / model / opt` + `gc.collect()` after each cohort | Cell 3/5 |
| `workers=0` in DataLoader | Cell 4 |
| State dict stored `.cpu().clone()` only | Cell 7 |
| Sequential cohort loop — only one cohort in RAM at a time | Cell 3 |
| Figures generated and immediately closed per cohort | Cell 6 |
| Full model **not** kept in `ALL_RESULTS` after export | Cell 7 |

**Writes (canonical NB01 layout):**
- `checkpoints/NB01/{cohort}_nb01_dae_joint.pt`
- `embeddings/NB01/{cohort}_nb01_isolated_latents.csv`
- `embeddings/NB01/{cohort}_nb01_joint_latents.csv`
- `results/NB01/figures/NB01_{cohort}_*.png`


### Cell 1 — Imports & Seeding
Locks all RNG states. Sets `NB_VERSION` and `PIPELINE_STAGE` constants.

In [1]:
# ==============================================================================
# CELL 1: IMPORTS & DETERMINISTIC SEEDING
# ==============================================================================
import os, gc, random, warnings, platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # no GUI — saves RAM on headless machines
import matplotlib.pyplot as plt
import yaml, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore")

NB_VERSION     = "1.0-ram"
PIPELINE_STAGE = "NB01"
SEED           = 42

def enforce_determinism(seed: int = 42) -> None:
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

enforce_determinism(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ {PIPELINE_STAGE} v{NB_VERSION} | Device: {device} | Seed: {SEED}")
print(f"   Python  : {platform.python_version()}")
print(f"   PyTorch : {torch.__version__}")


✅ NB01 v1.0-ram | Device: cpu | Seed: 42
   Python  : 3.12.3
   PyTorch : 2.12.0+cpu


### Cell 2 — Config & Canonical Paths
Loads `config.yaml` and resolves every input/output path following the NB01 canonical layout.

In [2]:
# ==============================================================================
# CELL 2: CONFIG & CANONICAL PATHS  (NB01 layout)
# ==============================================================================
CONFIG_PATH = Path("config.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError("❌ 'config.yaml' missing from workspace root.")

with open(CONFIG_PATH, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

BASE_DIR = Path.cwd()

pp = cfg.get("preprocessing", {})
m  = cfg.get("model",         {})
t  = cfg.get("training",      {})
s4 = cfg.get("stage4_gcn",    {})

# ── Hyperparameters ───────────────────────────────────────────────────────────
TARGET_FEATURES = pp.get("variance_filter_top_n",       3000)
LATENT_DIM      = m.get("latent_dim",    s4.get("latent_dim",    64))
NUM_INTERVALS   = m.get("num_intervals", s4.get("num_intervals", 10))
ENCODER_HIDDEN  = m.get("encoder_hidden",             [512, 256])
DROPOUT_RATE    = m.get("dropout_rate",                     0.40)
EPOCHS_DAE      = t.get("epochs",                            40)
EPOCHS_COX      = t.get("cox_epochs",                        80)
BATCH_SIZE      = t.get("batch_size",                        64)
LR              = t.get("lr",                              5e-4)
WEIGHT_DECAY    = t.get("weight_decay",                    0.01)
NOISE_FACTOR    = t.get("noise_factor",                    0.25)
COX_W           = t.get("cox_loss_weight",                  1.0)
STRUCT_W        = t.get("structural_loss_weight",            0.5)
COS_W           = t.get("cosine_loss_weight",                0.5)
PATIENCE        = t.get("patience",                            8)

# ── Canonical output dirs ─────────────────────────────────────────────────────
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / PIPELINE_STAGE
FIGURES_DIR    = BASE_DIR / "results"     / PIPELINE_STAGE / "figures"
EMBEDDINGS_DIR = BASE_DIR / "embeddings"  / PIPELINE_STAGE
RAW_DIR        = BASE_DIR / cfg.get("data", {}).get("raw_dir", "data/raw")

for _d in [CHECKPOINT_DIR, FIGURES_DIR, EMBEDDINGS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Multi-cohort config ───────────────────────────────────────────────────────
COHORT_CONFIGS = [
    {"name": "TCGA-LGG",  "short": "lgg",  "label": "Brain LGG",
     "expr": RAW_DIR / "TCGA-LGG.star_fpkm.tsv",  "surv": RAW_DIR / "TCGA-LGG.survival.tsv"},
    {"name": "TCGA-KIRC", "short": "kirc", "label": "Kidney KIRC",
     "expr": RAW_DIR / "TCGA-KIRC.star_fpkm.tsv", "surv": RAW_DIR / "TCGA-KIRC.survival.tsv"},
    {"name": "TCGA-LUAD", "short": "luad", "label": "Lung LUAD",
     "expr": RAW_DIR / "TCGA-LUAD.star_fpkm.tsv", "surv": RAW_DIR / "TCGA-LUAD.survival.tsv"},
]

print(f"🚀 {PIPELINE_STAGE} v{NB_VERSION}  |  {cfg['project']['name']} v{cfg['project']['version']}")
print(f"   Checkpoints : checkpoints/{PIPELINE_STAGE}/")
print(f"   Figures     : results/{PIPELINE_STAGE}/figures/")
print(f"   Embeddings  : embeddings/{PIPELINE_STAGE}/")
print(f"   Cohorts     : {[c['short'].upper() for c in COHORT_CONFIGS]}")
print(f"   Model       : latent={LATENT_DIM} | intervals={NUM_INTERVALS} | hidden={ENCODER_HIDDEN}")
print(f"   Training    : dae={EPOCHS_DAE} | cox={EPOCHS_COX} | batch={BATCH_SIZE} | lr={LR}")


🚀 NB01 v1.0-ram  |  universal-survival-engine v3.3
   Checkpoints : checkpoints/NB01/
   Figures     : results/NB01/figures/
   Embeddings  : embeddings/NB01/
   Cohorts     : ['LGG', 'KIRC', 'LUAD']
   Model       : latent=64 | intervals=10 | hidden=[512, 256]
   Training    : dae=40 | cox=80 | batch=64 | lr=0.0005


### Cell 3 — Data Loading · Variance Filter · Split · Scale
Per-cohort RAM-safe load. Runs inside the main cohort loop.

In [3]:
# ==============================================================================
# CELL 3: DATA LOADING — VARIANCE FILTER · SPLIT · SCALE
# (called once per cohort inside the main loop — Cell 5)
# ==============================================================================
def _trim(s: str) -> str:
    s = s.replace(".", "-").upper()
    return "-".join(s.split("-")[:3]) if s.startswith("TCGA") else s

def load_cohort(cc: dict, top_n: int = TARGET_FEATURES):
    """Load one cohort, variance-filter, split 60/20/20, scale. Returns dict."""
    print(f"  ↳ Loading {cc['name']} expression …")
    df_expr = pd.read_csv(cc["expr"], sep="\t", index_col=0)
    df_surv = pd.read_csv(cc["surv"], sep="\t", index_col=0)

    df_expr.columns = pd.Index([_trim(c) for c in df_expr.columns])
    df_surv.index   = pd.Index([_trim(i) for i in df_surv.index])
    df_expr = df_expr.loc[:, ~df_expr.columns.duplicated(keep="first")]
    df_surv = df_surv[~df_surv.index.duplicated(keep="first")]

    common = df_expr.columns.intersection(df_surv.index)
    if len(common) == 0:
        raise ValueError(f"Zero matching barcodes for {cc['name']}")
    print(f"     Aligned patients: {len(common)}")

    # Variance filter on float32 — avoids 64-bit RAM spike
    expr_t = df_expr[common].T.astype("float32")
    variances = expr_t.var(axis=0)
    top_genes = variances.nlargest(top_n).index.tolist()
    X_raw = expr_t[top_genes].values  # (patients × top_n)
    del df_expr, expr_t, variances
    gc.collect()

    # Survival labels
    time_col  = next((c for c in ["OS.time","survival_time","observed_time","time"]  if c in df_surv.columns), None)
    event_col = next((c for c in ["OS","event","observed_event","status"]             if c in df_surv.columns), None)
    if not time_col or not event_col:
        raise KeyError(f"Cannot find survival columns in {cc['name']}: {df_surv.columns.tolist()}")

    y_time  = df_surv.loc[common, time_col].values.astype("float32")
    y_event = df_surv.loc[common, event_col].values.astype("float32")
    del df_surv; gc.collect()

    # Stratified 60/20/20 split
    X_tr_r, X_te_r, t_tr, t_te, e_tr, e_te = train_test_split(
        X_raw, y_time, y_event,
        test_size=0.20, stratify=y_event.astype(int), random_state=SEED)
    X_tr_r, X_va_r, t_tr, t_va, e_tr, e_va = train_test_split(
        X_tr_r, t_tr, e_tr,
        test_size=0.25, stratify=e_tr.astype(int), random_state=SEED)

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_tr_r).astype("float32")
    X_val   = scaler.transform(X_va_r).astype("float32")
    X_test  = scaler.transform(X_te_r).astype("float32")
    del X_tr_r, X_va_r, X_te_r; gc.collect()

    print(f"     Train/Val/Test: {X_train.shape[0]}/{X_val.shape[0]}/{X_test.shape[0]}"
          f"  |  Features: {X_train.shape[1]}")

    return dict(
        X_train=X_train, X_val=X_val, X_test=X_test,
        t_tr=t_tr, t_va=t_va, t_te=t_te,
        e_tr=e_tr, e_va=e_va, e_te=e_te,
        X_raw=X_raw, y_time=y_time, y_event=y_event,
        top_genes=top_genes, scaler=scaler,
        input_dim=X_train.shape[1],
    )

print("✅ load_cohort() defined — called per cohort in Cell 5.")


✅ load_cohort() defined — called per cohort in Cell 5.


### Cell 4 — Dataset · Architecture · Loss
All class/function definitions. No cohort data loaded here.

In [4]:
# ==============================================================================
# CELL 4: DATASET · ARCHITECTURE · LOSS DEFINITIONS
# ==============================================================================

# ── Dataset ───────────────────────────────────────────────────────────────────
class SurvivalDataset(Dataset):
    def __init__(self, X, times, events):
        self.X = torch.tensor(X,      dtype=torch.float32)
        self.t = torch.tensor(times,  dtype=torch.float32)
        self.e = torch.tensor(events, dtype=torch.float32)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.t[i], self.e[i]

# ── Architecture ──────────────────────────────────────────────────────────────
class DAEEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim=64, hidden=None, dropout=0.40):
        super().__init__()
        hidden = hidden or [512, 256]
        layers, in_d = [], input_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers += [nn.Linear(in_d, latent_dim), nn.BatchNorm1d(latent_dim), nn.ReLU()]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class DAEDecoder(nn.Module):
    def __init__(self, latent_dim=64, output_dim=3000, hidden=None, dropout=0.40):
        super().__init__()
        hidden = hidden or [256, 512]
        layers, in_d = [], latent_dim
        for h in hidden:
            layers += [nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            in_d = h
        layers += [nn.Linear(in_d, output_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, z): return self.net(z)

class DiscreteHazardHead(nn.Module):
    """Discrete-time hazard head — replaces scalar Cox risk score."""
    def __init__(self, latent_dim=64, num_intervals=10, dropout=0.40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(latent_dim // 2, num_intervals), nn.Sigmoid(),
        )
    def forward(self, z): return self.net(z)   # (B, num_intervals)

class CoxLinearHead(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 1, bias=False)
    def forward(self, z): return self.fc(z).squeeze(-1)

# ── Losses ────────────────────────────────────────────────────────────────────
def cox_partial_loss(risk, times, events, eps=1e-7):
    """Breslow approximation of the Cox partial likelihood."""
    order   = torch.argsort(times, descending=True)
    risk    = risk[order]; events = events[order]
    log_cum = torch.logcumsumexp(risk, dim=0)
    loss    = -((risk - log_cum) * events).sum() / (events.sum() + eps)
    return loss

mse_criterion  = nn.MSELoss()
cosine_sim_fn  = nn.CosineSimilarity(dim=1)

print("✅ Dataset, architecture and loss functions defined.")


✅ Dataset, architecture and loss functions defined.


### Cell 5 — Main Cohort Training Loop
One cohort at a time: load → train → eval → export → del+gc.

In [5]:
# ==============================================================================
# CELL 5: MAIN COHORT LOOP — TRAIN · EVAL · EXPORT
# RAM: one cohort in memory at a time; full del+gc between cohorts.
# ==============================================================================
ALL_RESULTS = {}   # lightweight summary only — no tensors kept

for cc in COHORT_CONFIGS:
    cname = cc["name"]; sh = cc["short"]
    print(f"\n{'='*65}")
    print(f"🔬 Cohort: {cname}  ({cc['label']})")
    print(f"{'='*65}")

    # ── A. Load data ──────────────────────────────────────────────────────────
    data = load_cohort(cc)
    INPUT_DIM = data["input_dim"]

    train_ldr = DataLoader(
        SurvivalDataset(data["X_train"], data["t_tr"], data["e_tr"]),
        batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0)
    val_ldr   = DataLoader(
        SurvivalDataset(data["X_val"],   data["t_va"], data["e_va"]),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_ldr  = DataLoader(
        SurvivalDataset(data["X_test"],  data["t_te"], data["e_te"]),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # ── B. Build models ───────────────────────────────────────────────────────
    encoder  = DAEEncoder(INPUT_DIM, LATENT_DIM, ENCODER_HIDDEN, DROPOUT_RATE).to(device)
    decoder  = DAEDecoder(LATENT_DIM, INPUT_DIM, ENCODER_HIDDEN[::-1], DROPOUT_RATE).to(device)
    cox_iso  = CoxLinearHead(LATENT_DIM).to(device)
    cox_jnt  = CoxLinearHead(LATENT_DIM).to(device)

    # ── C. Phase 1: DAE pretraining ───────────────────────────────────────────
    p1_opt = optim.AdamW(
        list(encoder.parameters()) + list(decoder.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    print(f"\n🟡 Phase 1 — DAE Pretraining | {EPOCHS_DAE} epochs")
    print("-" * 60)
    losses_p1 = []
    for epoch in range(1, EPOCHS_DAE + 1):
        encoder.train(); decoder.train()
        ep_loss, n = 0.0, 0
        for bx, _, _ in train_ldr:
            bx    = bx.to(device)
            noise = bx + NOISE_FACTOR * torch.randn_like(bx)
            p1_opt.zero_grad()
            recon = decoder(encoder(noise))
            loss  = mse_criterion(recon, bx)
            loss.backward(); p1_opt.step()
            ep_loss += loss.item() * bx.size(0); n += bx.size(0)
        avg = ep_loss / n; losses_p1.append(avg)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{EPOCHS_DAE} | Reconstruction MSE: {avg:.5f}")
    del p1_opt; gc.collect()
    print(f"✅ Phase 1 done. Final MSE: {losses_p1[-1]:.5f}")

    # ── D. Phase 2a: Isolated Cox ─────────────────────────────────────────────
    iso_opt = optim.AdamW(
        list(encoder.parameters()) + list(cox_iso.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    print(f"\n🔴 Phase 2a — Isolated Cox | {EPOCHS_COX} epochs")
    print("-" * 60)
    losses_iso = []
    for epoch in range(1, EPOCHS_COX + 1):
        encoder.train(); cox_iso.train()
        ep_loss, n = 0.0, 0
        for bx, bt, be in train_ldr:
            bx = bx.to(device); bt = bt.to(device); be = be.to(device)
            iso_opt.zero_grad()
            loss = cox_partial_loss(cox_iso(encoder(bx)), bt, be)
            loss.backward(); iso_opt.step()
            ep_loss += loss.item() * bx.size(0); n += bx.size(0)
        avg = ep_loss / n; losses_iso.append(avg)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{EPOCHS_COX} | Cox Loss: {avg:.5f}")
    del iso_opt; gc.collect()
    print(f"✅ Phase 2a done. Final loss: {losses_iso[-1]:.5f}")

    # ── E. Phase 2b: Joint multi-task ─────────────────────────────────────────
    jnt_opt = optim.AdamW(
        list(encoder.parameters()) + list(decoder.parameters()) + list(cox_jnt.parameters()),
        lr=LR, weight_decay=WEIGHT_DECAY)
    print(f"\n🟣 Phase 2b — Joint Multi-Task | {EPOCHS_COX} epochs")
    print(f"   Weights → Cox: {COX_W}  Structural: {STRUCT_W}  Cosine sub-weight: {COS_W}")
    print("-" * 60)
    epochs_jnt, cox_jnt_l, struct_jnt_l, total_jnt_l = [], [], [], []
    for epoch in range(1, EPOCHS_COX + 1):
        encoder.train(); decoder.train(); cox_jnt.train()
        ep_cox = ep_struct = ep_total = 0.0; n = 0
        for bx, bt, be in train_ldr:
            bx = bx.to(device); bt = bt.to(device); be = be.to(device)
            jnt_opt.zero_grad()
            z = encoder(bx); recon = decoder(z); risk = cox_jnt(z)
            l_cox    = cox_partial_loss(risk, bt, be)
            l_mse    = mse_criterion(recon, bx)
            l_cos    = 1.0 - cosine_sim_fn(recon, bx).mean()
            l_struct = l_mse + COS_W * l_cos
            l_total  = COX_W * l_cox + STRUCT_W * l_struct
            l_total.backward(); jnt_opt.step()
            n += bx.size(0)
            ep_cox    += l_cox.item()    * bx.size(0)
            ep_struct += l_struct.item() * bx.size(0)
            ep_total  += l_total.item()  * bx.size(0)
        epochs_jnt.append(epoch)
        cox_jnt_l.append(ep_cox / n); struct_jnt_l.append(ep_struct / n); total_jnt_l.append(ep_total / n)
        if epoch % 20 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d}/{EPOCHS_COX} | Cox: {cox_jnt_l[-1]:.4f}"
                  f"  Struct: {struct_jnt_l[-1]:.4f}  Total: {total_jnt_l[-1]:.4f}")
    del jnt_opt; gc.collect()
    print(f"✅ Phase 2b done. Final total loss: {total_jnt_l[-1]:.4f}")

    # ── F. Eval helpers ───────────────────────────────────────────────────────
    def _preds(enc, head, ldr):
        enc.eval(); head.eval()
        all_t, all_e, all_r = [], [], []
        with torch.no_grad():
            for bx, bt, be in ldr:
                all_t.append(bt.numpy()); all_e.append(be.numpy())
                all_r.append(head(enc(bx.to(device))).cpu().numpy())
        return np.concatenate(all_t).ravel(), np.concatenate(all_e).ravel(), np.concatenate(all_r).ravel()

    def _ci(times, events, risks, label, n_boot=1000):
        mask = np.isfinite(times) & np.isfinite(events) & np.isfinite(risks)
        times, events, risks = times[mask], events[mask], risks[mask]
        if len(times) == 0 or events.sum() == 0:
            print(f"   ❌ [{label}] No valid samples."); return None
        ci = concordance_index(times, -risks, events)
        rng = np.random.default_rng(SEED); boot = []
        idx = np.arange(len(times))
        for _ in range(n_boot):
            bi = rng.choice(idx, size=len(times), replace=True)
            if events[bi].sum() == 0: continue
            try: boot.append(concordance_index(times[bi], -risks[bi], events[bi]))
            except: continue
        if boot:
            lo, hi = np.percentile(boot, [2.5, 97.5]); sd = np.std(boot)
            print(f"   ✅ [{label}] C-Index: {ci:.4f} | 95% CI [{lo:.4f}–{hi:.4f}] | SD ±{sd:.4f}")
            return {"c_index": ci, "lo": lo, "hi": hi, "sd": sd}
        print(f"   ✅ [{label}] C-Index: {ci:.4f}  (bootstrap failed)")
        return {"c_index": ci, "lo": None, "hi": None, "sd": None}

    print(f"\n📊 [{cname}] Evaluation")
    val_t_i,  val_e_i,  val_r_i  = _preds(encoder, cox_iso, val_ldr)
    test_t_i, test_e_i, test_r_i = _preds(encoder, cox_iso, test_ldr)
    val_t_j,  val_e_j,  val_r_j  = _preds(encoder, cox_jnt, val_ldr)
    test_t_j, test_e_j, test_r_j = _preds(encoder, cox_jnt, test_ldr)

    ci_iso_val  = _ci(val_t_i,  val_e_i,  val_r_i,  f"Val  Isolated [{sh.upper()}]")
    ci_iso_test = _ci(test_t_i, test_e_i, test_r_i, f"Test Isolated [{sh.upper()}]")
    ci_jnt_val  = _ci(val_t_j,  val_e_j,  val_r_j,  f"Val  Joint    [{sh.upper()}]")
    ci_jnt_test = _ci(test_t_j, test_e_j, test_r_j, f"Test Joint    [{sh.upper()}]")

    # ── G. Figures ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(range(1, len(losses_p1)+1), losses_p1, color="#888888", lw=2)
    ax.set_title(f"Phase 1 DAE Loss — {cname}", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE"); ax.grid(True, linestyle=":", alpha=0.6)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"NB01_{sh}_dae_loss.png", dpi=150, bbox_inches="tight")
    plt.close(); gc.collect()

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(epochs_jnt, cox_jnt_l,    color="#FF8C00", lw=2,   label="Cox")
    ax.plot(epochs_jnt, struct_jnt_l, color="#8A2BE2", lw=2, ls="--", label="Structural")
    ax.plot(epochs_jnt, total_jnt_l,  color="#000000", lw=2.5, label="Total")
    ax.set_title(f"Phase 2b Joint Loss — {cname}", fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(True, linestyle=":", alpha=0.6)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"NB01_{sh}_joint_loss.png", dpi=150, bbox_inches="tight")
    plt.close(); gc.collect()

    # Train-set predictions for KM thresholds (zero test leakage)
    train_t_i, train_e_i, train_r_i = _preds(encoder, cox_iso, train_ldr)
    train_t_j, train_e_j, train_r_j = _preds(encoder, cox_jnt, train_ldr)

    # KM — test cohort
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    for ax, r, t_, e_, label, colors in [
        (ax1, test_r_i, test_t_i, test_e_i, "Isolated Cox",    ("#DC143C", "#008080")),
        (ax2, test_r_j, test_t_j, test_e_j, "Joint Multi-Task",("#FF8C00", "#8A2BE2")),
    ]:
        thresh = np.median(train_r_i if "Isolated" in label else train_r_j)
        for mask_, lbl_, c_ in [(r >= thresh, "High Risk", colors[0]), (r < thresh, "Low Risk", colors[1])]:
            if mask_.sum() > 1:
                KaplanMeierFitter().fit(t_[mask_], e_[mask_], label=f"{lbl_} (n={mask_.sum()})").plot(ax=ax, color=c_, lw=2)
        ax.set_title(f"{label} — {cname}", fontweight="bold")
        ax.set_xlabel("Days"); ax.grid(True, linestyle=":", alpha=0.5); ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"NB01_{sh}_km.png", dpi=150, bbox_inches="tight")
    plt.close(); gc.collect()

    # ── H. Checkpoint & embeddings ────────────────────────────────────────────
    ckpt_name = f"{sh}_nb01_dae_joint.pt"
    torch.save({
        "encoder_state":      encoder.state_dict(),
        "decoder_state":      decoder.state_dict(),
        "cox_iso_state":      cox_iso.state_dict(),
        "cox_jnt_state":      cox_jnt.state_dict(),
        "config": {
            "input_dim":  INPUT_DIM, "latent_dim": LATENT_DIM,
            "hidden":     ENCODER_HIDDEN, "dropout": DROPOUT_RATE,
            "num_intervals": NUM_INTERVALS,
        },
        "honest_ci": {"c_index": ci_jnt_test.get("c_index") if ci_jnt_test else None},
        "top_genes":      data["top_genes"],
        "scaler_mean":    data["scaler"].mean_,
        "scaler_var":     data["scaler"].var_,
        "pipeline_stage": PIPELINE_STAGE,
        "nb_version":     NB_VERSION,
        "cohort":         cname,
    }, CHECKPOINT_DIR / ckpt_name)
    print(f"\n  💾 checkpoints/{PIPELINE_STAGE}/{ckpt_name}  ✅")

    # ── Embedding export ─────────────────────────────────────────────────
    z_cols = [f"z_{i}" for i in range(LATENT_DIM)]
    encoder.eval(); cox_iso.eval(); cox_jnt.eval()

    # — held-out: data["scaler"] was fit on train split only (leakage-safe) —
    # Use for any downstream survival metric or audit.
    X_ho_sc = data["scaler"].transform(data["X_raw"]).astype("float32")
    with torch.no_grad():
        X_t   = torch.tensor(X_ho_sc).to(device)
        z_ho  = encoder(X_t).cpu().numpy()
        z_t   = torch.tensor(z_ho).to(device)
        r_iso = cox_iso(z_t).cpu().numpy()
        r_jnt = cox_jnt(z_t).cpu().numpy()
        del z_t
    del X_ho_sc, X_t; gc.collect()

    df_iso_ho = pd.DataFrame(z_ho, columns=z_cols)
    df_iso_ho["risk_score_isolated"] = r_iso
    df_iso_ho["survival_time"] = data["y_time"]; df_iso_ho["event"] = data["y_event"]
    df_jnt_ho = pd.DataFrame(np.concatenate([z_ho, r_jnt.reshape(-1,1)], axis=1),
                             columns=z_cols + ["risk_score_joint"])
    df_jnt_ho["survival_time"] = data["y_time"]; df_jnt_ho["event"] = data["y_event"]
    df_iso_ho.to_csv(EMBEDDINGS_DIR / f"{sh}_nb01_isolated_heldout_latents.csv", index=False)
    df_jnt_ho.to_csv(EMBEDDINGS_DIR / f"{sh}_nb01_joint_heldout_latents.csv",    index=False)
    print(f"  📁 {sh}_nb01_isolated_heldout_latents.csv  (leakage-safe — train scaler)")
    print(f"  📁 {sh}_nb01_joint_heldout_latents.csv     (leakage-safe — train scaler)")
    del z_ho, r_iso, r_jnt, df_iso_ho, df_jnt_ho; gc.collect()

    # — full-cohort: scaler refit on all patients (representation only) —
    # Use for UMAP, clustering, cross-NB comparison. NOT for survival metrics.
    sc_full = StandardScaler()
    X_fc_sc = sc_full.fit_transform(data["X_raw"]).astype("float32")
    with torch.no_grad():
        X_t    = torch.tensor(X_fc_sc).to(device)
        z_fc   = encoder(X_t).cpu().numpy()
        z_t    = torch.tensor(z_fc).to(device)
        r_iso_fc = cox_iso(z_t).cpu().numpy()
        r_jnt_fc = cox_jnt(z_t).cpu().numpy()
        del z_t
    del X_fc_sc, X_t; gc.collect()

    df_iso_fc = pd.DataFrame(z_fc, columns=z_cols)
    df_iso_fc["risk_score_isolated"] = r_iso_fc
    df_iso_fc["survival_time"] = data["y_time"]; df_iso_fc["event"] = data["y_event"]
    df_jnt_fc = pd.DataFrame(np.concatenate([z_fc, r_jnt_fc.reshape(-1,1)], axis=1),
                             columns=z_cols + ["risk_score_joint"])
    df_jnt_fc["survival_time"] = data["y_time"]; df_jnt_fc["event"] = data["y_event"]
    df_iso_fc.to_csv(EMBEDDINGS_DIR / f"{sh}_nb01_isolated_fullcohort_latents.csv", index=False)
    df_jnt_fc.to_csv(EMBEDDINGS_DIR / f"{sh}_nb01_joint_fullcohort_latents.csv",    index=False)
    print(f"  📁 {sh}_nb01_isolated_fullcohort_latents.csv  (full-cohort scaler — representation only)")
    print(f"  📁 {sh}_nb01_joint_fullcohort_latents.csv     (full-cohort scaler — representation only)")
    del z_fc, r_iso_fc, r_jnt_fc, df_iso_fc, df_jnt_fc, sc_full; gc.collect()

    # ── I. Lightweight summary + full cleanup ─────────────────────────────────
    ALL_RESULTS[sh] = {
        "cohort": cname,
        "honest_ci_iso": ci_iso_test,
        "honest_ci_jnt": ci_jnt_test,
        "checkpoint": str(CHECKPOINT_DIR / ckpt_name),
    }

    del (encoder, decoder, cox_iso, cox_jnt,
         train_ldr, val_ldr, test_ldr,
         data, losses_p1, losses_iso,
         epochs_jnt, cox_jnt_l, struct_jnt_l, total_jnt_l)
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    print(f"  🧹 [{cname}] RAM released.")

print(f"\n{'='*65}")
print(f"✅ NB01 complete — all {len(ALL_RESULTS)} cohorts processed.")



🔬 Cohort: TCGA-LGG  (Brain LGG)
  ↳ Loading TCGA-LGG expression …
     Aligned patients: 516
     Train/Val/Test: 309/103/104  |  Features: 3000

🟡 Phase 1 — DAE Pretraining | 40 epochs
------------------------------------------------------------
  Epoch   1/40 | Reconstruction MSE: 1.23837
  Epoch  10/40 | Reconstruction MSE: 0.70684
  Epoch  20/40 | Reconstruction MSE: 0.61008
  Epoch  30/40 | Reconstruction MSE: 0.56619
  Epoch  40/40 | Reconstruction MSE: 0.52846
✅ Phase 1 done. Final MSE: 0.52846

🔴 Phase 2a — Isolated Cox | 80 epochs
------------------------------------------------------------
  Epoch   1/80 | Cox Loss: 2.84596
  Epoch  20/80 | Cox Loss: 1.26129
  Epoch  40/80 | Cox Loss: 0.92326
  Epoch  60/80 | Cox Loss: 0.80098
  Epoch  80/80 | Cox Loss: 0.79912
✅ Phase 2a done. Final loss: 0.79912

🟣 Phase 2b — Joint Multi-Task | 80 epochs
   Weights → Cox: 1.0  Structural: 0.5  Cosine sub-weight: 0.5
------------------------------------------------------------
  Epoch   1/8

### Cell 6 — Cross-Cohort Summary

In [6]:
# ==============================================================================
# CELL 6: CROSS-COHORT SUMMARY TABLE
# Requires: ALL_RESULTS populated (Cell 5 complete).
# ==============================================================================
print(f"\n{'='*65}")
print(f"  {'Cohort':<10} {'Isolated C-idx':>16}  {'Joint C-idx':>14}")
print(f"{'='*65}")
for sh, r in ALL_RESULTS.items():
    iso = r["honest_ci_iso"]; jnt = r["honest_ci_jnt"]
    iso_s = f"{iso['c_index']:.4f}" if iso else "N/A"
    jnt_s = f"{jnt['c_index']:.4f}" if jnt else "N/A"
    print(f"  {sh.upper():<10} {iso_s:>16}  {jnt_s:>14}")
print(f"{'='*65}")
print(f"\n  Handoff artefacts:")
for sh, r in ALL_RESULTS.items():
    print(f"   [{sh.upper()}] checkpoints/NB01/{sh}_nb01_dae_joint.pt")
    print(f"         embeddings/NB01/{sh}_nb01_isolated_latents.csv")
    print(f"         embeddings/NB01/{sh}_nb01_joint_latents.csv")



  Cohort       Isolated C-idx     Joint C-idx
  LGG                  0.8420          0.8481
  KIRC                 0.7088          0.7316
  LUAD                 0.5109          0.6026

  Handoff artefacts:
   [LGG] checkpoints/NB01/lgg_nb01_dae_joint.pt
         embeddings/NB01/lgg_nb01_isolated_latents.csv
         embeddings/NB01/lgg_nb01_joint_latents.csv
   [KIRC] checkpoints/NB01/kirc_nb01_dae_joint.pt
         embeddings/NB01/kirc_nb01_isolated_latents.csv
         embeddings/NB01/kirc_nb01_joint_latents.csv
   [LUAD] checkpoints/NB01/luad_nb01_dae_joint.pt
         embeddings/NB01/luad_nb01_isolated_latents.csv
         embeddings/NB01/luad_nb01_joint_latents.csv
